# AIDA 1141 Project Exercise 06 – Model Training and Split

**Question:** Can a logistic regression model trained only on training employees identify at-risk early-warning assessments with at least 90% recall while flagging fewer assessments than the Exercise 05 days-until-start rule?

This notebook uses the original synthetic files from Exercise 02, the label defined in Exercise 04, and the fixed employee-level split and baseline thresholds saved in Exercise 05. The model is fitted on training early-warning assessments only, and validation is used for comparison. Test data is included only in split-consistency checks, not model fitting, threshold selection, or performance comparison. Fuller evaluation measures follow in Exercise 07.

All results describe a model applied to simulated outcomes; they are not evidence about real employees or HR performance.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

EX02 = Path("../02-synthetic-request-records")
EX05 = Path("../05-simple-routing-baseline")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
SEED = 2026

read = lambda name: pd.read_csv(EX02 / "data" / name, keep_default_na=False, na_values=[""])
requests = read("requests.csv")
assessments = read("assessments.csv")
dictionary = pd.read_csv(EX02 / "data-dictionary.csv", keep_default_na=False)
employee_split = pd.read_csv(EX05 / "outputs" / "employee_split.csv")
rule_thresholds = pd.read_csv(EX05 / "outputs" / "rule_thresholds.csv")

request_inputs = dictionary.query("file == 'requests.csv' and role == 'model_input'")["field"].tolist()
data = (assessments.merge(requests[["request_id"] + request_inputs], on="request_id", how="left", validate="many_to_one")
        .merge(employee_split, on="employee_id", how="left", validate="many_to_one"))
data["early_warning"] = ~data["past_deadline_at_assessment"]
print(f"Loaded {len(data):,} assessments; scikit-learn {sklearn.__version__}")

Loaded 2,579 assessments; scikit-learn 1.9.1


## A. Data and split checks

The saved Exercise 05 split is reused unchanged, so the model and the baselines are compared on the same employees. These checks confirm that the split loaded correctly and matches Exercise 05.

In [2]:
early = data[data["early_warning"]]
split_counts = (early.groupby("split")
                .agg(employees=("employee_id", "nunique"), assessments=("at_risk", "size"), at_risk=("at_risk", "sum"))
                .reindex(["train", "validation", "test"]))
setup_checks = pd.Series({
    "Every assessment has a split": data["split"].notna().all(),
    "500 employees in the saved split": employee_split["employee_id"].is_unique and len(employee_split) == 500,
    "Split matches Exercise 05 (299 / 103 / 98 employees)":
        employee_split["split"].value_counts().reindex(["train", "validation", "test"]).tolist() == [299, 103, 98],
    "Early-warning at-risk counts match Exercise 05 (246 / 84 / 78)": split_counts["at_risk"].tolist() == [246, 84, 78],
    "One row per assessment": data["assessment_id"].is_unique,
    "Assessment and request employee IDs agree":
        (data["employee_id"] == data["request_id"].map(requests.set_index("request_id")["employee_id"])).all(),
}, name="passed")
assert setup_checks.all(), "The data or split does not match Exercise 05."
display(setup_checks.to_frame())
split_counts

,passed
Every assessment has a split,True
500 employees in the saved split,True
Split matches Exercise 05 (299 / 103 / 98 employees),True
Early-warning at-risk counts match Exercise 05 (246 / 84 / 78),True
One row per assessment,True
Assessment and request employee IDs agree,True


,employees,assessments,at_risk
split,,,
train,294,1509,246
validation,100,505,84
test,96,502,78


## B. Features and feature-level leakage checks

The model uses the documented `model_input` fields from the Exercise 02 data dictionary, which Exercise 04 reviewed for availability at assessment time:

- **Numeric:** business days until start, business days in current status, and team open workload. These are standardized inside the pipeline.
- **Categorical:** current status, task type, hiring department, employee type, intake channel, and information issue at intake. These are one-hot encoded inside the pipeline with `drop="first"`, so the first category of each is the reference level; the actual reference categories are saved after fitting.

`responsible_team` is also a `model_input` field but is left out: the routing rule sets it directly from `task_type`, so including both would duplicate the same information. IDs, the label, outcome fields, simulation-only fields, the split, and the stratification counts are never used as features.

The checks below cover the features and the employee split. Checks that fitting and preprocessing use training rows only are added in the cross-validation and final fitting sections, where that fitting happens.

In [3]:
NUMERIC = ["business_days_until_start", "business_days_in_current_status", "team_open_workload"]
CATEGORICAL = ["current_status", "task_type", "hiring_department", "employee_type", "intake_channel", "info_issue_at_intake"]
FEATURES = NUMERIC + CATEGORICAL
EXCLUDED_INPUTS = {"responsible_team": "set directly from task_type by the routing rule, so it duplicates task_type"}

# Roles from the two files the features come from
source_roles = dictionary[dictionary["file"].isin(["requests.csv", "assessments.csv"])]
roles = source_roles.drop_duplicates(["field", "role"]).set_index("field")["role"]
assert roles.index.is_unique, "A field has different roles in requests.csv and assessments.csv."
model_inputs = set(source_roles.loc[source_roles["role"] == "model_input", "field"])
not_allowed = set(source_roles.loc[source_roles["role"].isin(["id", "label", "outcome", "simulation_only"]), "field"])

leakage_checks = pd.Series({
    "Every feature has the model_input role": all(roles.get(f) == "model_input" for f in FEATURES),
    "Every model_input field is used or explained": model_inputs == set(FEATURES) | set(EXCLUDED_INPUTS),
    "No ID, label, outcome, or simulation-only field is a feature": not set(FEATURES) & not_allowed,
    "No missing values in the features": not data[FEATURES].isna().any().any(),
    "Split and stratification columns are not features": not set(FEATURES) & {"split", "late_tasks", "late_group"},
    "No employee appears in more than one split": data.groupby("employee_id")["split"].nunique().max() == 1,
}, name="passed")
assert leakage_checks.all(), "A leakage check failed."

feature_list = pd.DataFrame({
    "feature": FEATURES,
    "type": ["numeric"] * len(NUMERIC) + ["categorical"] * len(CATEGORICAL),
    "source": ["assessments.csv" if f in assessments.columns else "requests.csv" for f in FEATURES],
})
feature_list.to_csv(OUTPUT_DIR / "feature_list.csv", index=False)
display(leakage_checks.to_frame())
feature_list

,passed
Every feature has the model_input role,True
Every model_input field is used or explained,True
"No ID, label, outcome, or simulation-only field is a feature",True
No missing values in the features,True
Split and stratification columns are not features,True
No employee appears in more than one split,True


,feature,type,source
0,business_days_until_start,numeric,assessments.csv
1,business_days_in_current_status,numeric,assessments.csv
2,team_open_workload,numeric,assessments.csv
3,current_status,categorical,assessments.csv
4,task_type,categorical,requests.csv
5,hiring_department,categorical,requests.csv
6,employee_type,categorical,requests.csv
7,intake_channel,categorical,requests.csv
8,info_issue_at_intake,categorical,requests.csv


## C. Cross-validated threshold on training data

The model flags an assessment when its estimated risk probability is at or above a threshold. To choose that threshold without using validation data, the training employees are divided into five folds, keeping each employee's assessments in one fold. For each fold, the whole pipeline (scaling, encoding, and logistic regression) is fitted on the other four folds and predicts the held-out fold. Every training assessment therefore receives one out-of-fold prediction from a model that never saw that employee.

**Threshold rule:** choose the highest threshold that still achieves at least 90% recall on the out-of-fold predictions. Because the model flags scores at or above the threshold, a higher threshold flags fewer assessments, so this gives the fewest flags that meet the target. This uses the same training-only principle as the Exercise 05 baselines, although the procedures differ: the baseline thresholds used training observations directly, while this threshold uses out-of-fold model scores. In Exercise 09, both the model and the baseline rules can be re-tuned on validation before anything is frozen for the final test.

The logistic regression uses scikit-learn's defaults (L2 regularization, C = 1.0). The model is unweighted. The threshold determines which assessments are flagged, and its observed recall is measured on the out-of-fold predictions. Fold assignment uses seed 2026. The threshold is kept at full precision for later predictions; printed values are rounded. The fold summary and out-of-fold predictions are saved so the selection can be reviewed.

In [4]:
RECALL_TARGET = 0.90
N_FOLDS = 5

def make_pipeline():
    """Preprocessing and logistic regression in one pipeline, so every fit uses only the rows it is given."""
    prepare = ColumnTransformer([
        ("numeric", StandardScaler(), NUMERIC),
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore"), CATEGORICAL),
    ])
    return Pipeline([("prepare", prepare), ("model", LogisticRegression(max_iter=1000))])

def model_frame(rows):
    """Feature columns, with categories (including True/False) treated as text labels."""
    return rows[FEATURES].astype({c: str for c in CATEGORICAL})

train = early[early["split"] == "train"]
X_train, y_train = model_frame(train), train["at_risk"].astype(int)
assert y_train.sum() > 0, "Training data contains no at-risk assessments."

folds = GroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
out_of_fold = pd.Series(np.nan, index=train.index)
prediction_count = np.zeros(len(train), dtype=int)
fold_rows = []
for fold, (fit_idx, held_idx) in enumerate(folds.split(X_train, y_train, groups=train["employee_id"]), start=1):
    assert y_train.iloc[fit_idx].nunique() == 2, "A fitting fold does not contain both outcome classes."
    fitted = make_pipeline().fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
    out_of_fold.iloc[held_idx] = fitted.predict_proba(X_train.iloc[held_idx])[:, 1]
    prediction_count[held_idx] += 1
    fold_rows.append({
        "fold": fold, "fit_assessments": len(fit_idx), "fit_at_risk": int(y_train.iloc[fit_idx].sum()),
        "held_out_assessments": len(held_idx), "held_out_at_risk": int(y_train.iloc[held_idx].sum()),
        "employees_in_both": len(set(train["employee_id"].iloc[fit_idx]) & set(train["employee_id"].iloc[held_idx])),
    })
fold_summary = pd.DataFrame(fold_rows)

cv_checks = pd.Series({
    "Cross-validation uses training employees only": train["split"].eq("train").all(),
    "Every training assessment predicted exactly once": (prediction_count == 1).all() and out_of_fold.notna().all(),
    "No employee is in both parts of any fold": fold_summary["employees_in_both"].eq(0).all(),
    "Every score is a finite value between 0 and 1": np.isfinite(out_of_fold).all() and out_of_fold.between(0, 1).all(),
}, name="passed")
assert cv_checks.all(), "A cross-validation check failed."

def recall_at(scores, labels, threshold):
    """Share of at-risk assessments with a score at or above the threshold."""
    return ((scores >= threshold) & (labels == 1)).sum() / (labels == 1).sum()

# Highest threshold that still reaches the recall target, so the fewest assessments are flagged
candidates = np.sort(out_of_fold.unique())
qualifying = [t for t in candidates if recall_at(out_of_fold, y_train, t) >= RECALL_TARGET]
model_threshold = float(max(qualifying))   # kept at full precision; the printout below is rounded

fold_summary.to_csv(OUTPUT_DIR / "cv_fold_summary.csv", index=False)
(train.assign(out_of_fold_probability=out_of_fold)
      [["assessment_id", "request_id", "employee_id", "at_risk", "out_of_fold_probability"]]
      .to_csv(OUTPUT_DIR / "train_out_of_fold_predictions.csv", index=False))

display(cv_checks.to_frame())
display(fold_summary)
print(f"Selected threshold: flag when estimated risk probability >= {model_threshold:.4f} (rounded for display)")
print(f"Out-of-fold training recall: {recall_at(out_of_fold, y_train, model_threshold):.1%}; "
      f"assessments flagged: {(out_of_fold >= model_threshold).mean():.1%}")

,passed
Cross-validation uses training employees only,True
Every training assessment predicted exactly once,True
No employee is in both parts of any fold,True
Every score is a finite value between 0 and 1,True


,fold,fit_assessments,fit_at_risk,held_out_assessments,held_out_at_risk,employees_in_both
0,1,1223,203,286,43,0
1,2,1205,185,304,61,0
2,3,1207,208,302,38,0
3,4,1206,186,303,60,0
4,5,1195,202,314,44,0


Selected threshold: flag when estimated risk probability >= 0.1254 (rounded for display)
Out-of-fold training recall: 90.2%; assessments flagged: 32.5%


## D. Final model, coefficients, and saved settings

A check confirms that every row supplied for fitting belongs to the training split; the same pipeline is then refitted once on all training early-warning assessments and saved with the threshold chosen in Part C.

**Coefficients** are on the log-odds scale. Numeric coefficients describe a one-standard-deviation increase, because those inputs are standardized; the training means and standard deviations are saved in `numeric_scaling.csv`. Categorical coefficients compare each category with its reference category, which is listed beside it. Because these describe different kinds of change, numeric and categorical coefficients should not be ranked against each other by size. Each coefficient describes an association with the simulated outcome after accounting for the other inputs, so it may differ from the single-input patterns in Exercise 04; this analysis does not establish exactly why. Coefficients describe associations in the simulation, not causes of delays, and regularization does not make them causal.

The model file alone does not record the flagging rule, so `model_settings.csv` saves the threshold at full precision together with the features, preprocessing, reference categories, cross-validation settings, seed, and package versions. Each check result is saved in `leakage_checks.csv`. Loading `model.joblib` requires matching package versions.

In [5]:
import scipy

# Final model: the same pipeline refitted once on all training early-warning assessments
assert data.loc[X_train.index, "split"].eq("train").all(), "Non-training rows supplied for fitting."
model = make_pipeline().fit(X_train, y_train)

# Coefficients, with the reference category each one-hot column is compared against
encoder = model.named_steps["prepare"].named_transformers_["categorical"]
reference = {feature: str(categories[0]) for feature, categories in zip(CATEGORICAL, encoder.categories_)}
names = model.named_steps["prepare"].get_feature_names_out()
coefficients = pd.DataFrame({"term": names, "coefficient": model.named_steps["model"].coef_[0]})
coefficients["feature"] = [next(f for f in FEATURES if n.split("__", 1)[1].startswith(f)) for n in names]
coefficients["compared_with"] = ["1 standard deviation increase" if f in NUMERIC else f"reference: {reference[f]}"
                                 for f in coefficients["feature"]]
coefficients = pd.concat([
    pd.DataFrame([{"term": "intercept", "coefficient": model.named_steps["model"].intercept_[0],
                   "feature": "intercept", "compared_with": "all inputs at reference level or training mean"}]),
    coefficients.sort_values("coefficient"),
], ignore_index=True)
coefficients.round(4).to_csv(OUTPUT_DIR / "coefficients.csv", index=False)

# Scaling used for the numeric inputs, so coefficients can be interpreted without loading the model
scaler = model.named_steps["prepare"].named_transformers_["numeric"]
numeric_scaling = pd.DataFrame({"feature": NUMERIC, "training_mean": scaler.mean_,
                                "training_standard_deviation": scaler.scale_})
numeric_scaling.round(4).to_csv(OUTPUT_DIR / "numeric_scaling.csv", index=False)

# Save the fitted pipeline, each check, and every setting needed to reproduce the full flagging rule
joblib.dump(model, OUTPUT_DIR / "model.joblib")
all_checks = pd.concat({"data and split": setup_checks, "features": leakage_checks, "cross-validation": cv_checks})
all_checks.rename_axis(["stage", "check"]).to_csv(OUTPUT_DIR / "leakage_checks.csv")
model_settings = pd.Series({
    "model": "LogisticRegression (L2, C = 1.0, unweighted, max_iter = 1000)",
    "package_versions": (f"scikit-learn {sklearn.__version__}; pandas {pd.__version__}; NumPy {np.__version__}; "
                         f"joblib {joblib.__version__}; SciPy {scipy.__version__}"),
    "features": ", ".join(FEATURES),
    "excluded_model_input": "responsible_team (duplicates task_type)",
    "numeric_preprocessing": "StandardScaler",
    "categorical_preprocessing": "OneHotEncoder(drop='first', handle_unknown='ignore')",
    "reference_categories": "; ".join(f"{f} = {r}" for f, r in reference.items()),
    "training_rows": f"{len(X_train)} training early-warning assessments ({int(y_train.sum())} at risk)",
    "cross_validation": f"GroupKFold, {N_FOLDS} folds grouped by employee, shuffle with seed {SEED}",
    "recall_target": RECALL_TARGET,
    "threshold_rule": "highest threshold with at least 90% out-of-fold training recall; flag when probability >= threshold",
    "threshold": repr(model_threshold),
    "checks_passed": f"{int(all_checks.sum())} of {len(all_checks)}",
}, name="value")
model_settings.to_csv(OUTPUT_DIR / "model_settings.csv", header=True)

display(model_settings.to_frame())
display(numeric_scaling.round(3))
coefficients.round(3)

,value
model,"LogisticRegression (L2, C = 1.0, unweighted, m..."
package_versions,scikit-learn 1.9.1; pandas 3.0.6; NumPy 2.5.3;...
features,"business_days_until_start, business_days_in_cu..."
excluded_model_input,responsible_team (duplicates task_type)
numeric_preprocessing,StandardScaler
categorical_preprocessing,"OneHotEncoder(drop='first', handle_unknown='ig..."
reference_categories,current_status = in_progress; task_type = equi...
training_rows,1509 training early-warning assessments (246 a...
cross_validation,"GroupKFold, 5 folds grouped by employee, shuff..."
recall_target,0.9


,feature,training_mean,training_standard_deviation
0,business_days_until_start,11.406,7.476
1,business_days_in_current_status,0.581,1.418
2,team_open_workload,18.604,12.549


,term,coefficient,feature,compared_with
0,intercept,-4.855,intercept,all inputs at reference level or training mean
1,numeric__business_days_until_start,-5.172,business_days_until_start,1 standard deviation increase
2,categorical__task_type_orientation_scheduling,-1.123,task_type,reference: equipment_setup
3,categorical__hiring_department_Parks,-1.099,hiring_department,reference: Administration
4,categorical__task_type_system_access,-1.003,task_type,reference: equipment_setup
5,categorical__hiring_department_Finance,-0.735,hiring_department,reference: Administration
6,categorical__hiring_department_Transit,-0.598,hiring_department,reference: Administration
7,categorical__hiring_department_Recreation,-0.515,hiring_department,reference: Administration
8,numeric__business_days_in_current_status,-0.378,business_days_in_current_status,1 standard deviation increase
9,categorical__hiring_department_Utilities,-0.320,hiring_department,reference: Administration


## E. Validation comparison with the Exercise 05 baselines

The final model scores every validation assessment, and assessments at or above the threshold are flagged. The Exercise 05 rules are re-applied using their saved thresholds, and every method is evaluated with the same function on the 505 validation early-warning assessments. The model was not fitted or tuned on validation data.

Recall is the main measure because of the 90% target. Flagged assessments and unique tasks flagged are reported separately; neither measures staff time. The validation set is small and clustered by task and employee, so these results describe one comparison on this split rather than a reliable estimate of improvement. The 21 overdue validation assessments are reported separately. Fuller evaluation measures, including which errors matter most, follow in Exercise 07.

In [6]:
# Validation: estimated risk probabilities and flags for every validation assessment
validation = data[data["split"] == "validation"].copy()
validation["risk_probability"] = model.predict_proba(model_frame(validation))[:, 1]
validation["logistic_regression"] = validation["risk_probability"] >= model_threshold

# Exercise 05 rules, re-applied with their saved thresholds
global_threshold = int(rule_thresholds.query("employee_type == 'all'")["threshold"].item())
type_thresholds = rule_thresholds.query("employee_type != 'all'").set_index("employee_type")["threshold"]
assert validation["employee_type"].isin(type_thresholds.index).all(), "An employee type has no threshold."
validation["days_until_start_rule"] = validation["business_days_until_start"] <= global_threshold
validation["days_rule_by_employee_type"] = (validation["business_days_until_start"]
                                            <= validation["employee_type"].map(type_thresholds))

# The re-applied rules must reproduce the predictions saved in Exercise 05
ex05_predictions = pd.read_csv(EX05 / "outputs" / "baseline_predictions_validation.csv")
matched = validation.merge(ex05_predictions, on="assessment_id", suffixes=("", "_ex05"), validate="one_to_one")
assert len(matched) == len(validation), "Validation assessments do not match Exercise 05."
for rule in ["days_until_start_rule", "days_rule_by_employee_type"]:
    assert (matched[rule] == matched[f"{rule}_ex05"]).all(), f"{rule} does not match Exercise 05."
print("Re-applied rules match the Exercise 05 validation predictions")

def evaluate(group, flag):
    """Counts and measures for one method's flags on one group of assessments."""
    y, p = group["at_risk"], group[flag]
    tp, fp = (p & y).sum(), (p & ~y).sum()
    fn, tn = (~p & y).sum(), (~p & ~y).sum()
    return pd.Series({
        "assessments": len(group), "at_risk": y.sum(), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "recall": tp / (tp + fn) if tp + fn else np.nan, "precision": tp / (tp + fp) if tp + fp else np.nan,
        "flagged_assessments": p.sum(), "flagged_%": p.mean() * 100,
        "unique_tasks_flagged": group.loc[p, "request_id"].nunique(),
        "unique_tasks": group["request_id"].nunique(),
        "accuracy": (p == y).mean(),
    })

METHODS = ["logistic_regression", "days_until_start_rule", "days_rule_by_employee_type"]
validation_early = validation[validation["early_warning"]]
model_vs_baselines = pd.DataFrame({m: evaluate(validation_early, m) for m in METHODS}).T
model_vs_baselines = model_vs_baselines.astype({c: int for c in ["assessments", "at_risk", "TP", "FP", "FN", "TN",
                                                                  "flagged_assessments", "unique_tasks_flagged",
                                                                  "unique_tasks"]})
model_vs_baselines.round(3).to_csv(OUTPUT_DIR / "model_vs_baselines.csv")

# Overdue validation assessments are reported separately
overdue = validation[~validation["early_warning"]]
overdue_flags = pd.Series({m: int(overdue[m].sum()) for m in METHODS}, name=f"flagged (of {len(overdue)} overdue)")

prediction_columns = ["assessment_id", "request_id", "employee_id", "checkpoint", "early_warning", "at_risk",
                      "risk_probability"] + METHODS
validation[prediction_columns].to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
print(f"Saved predictions for {len(validation)} validation assessments")

display(overdue_flags.to_frame())
model_vs_baselines[["at_risk", "TP", "FN", "recall", "precision", "flagged_assessments", "flagged_%",
                    "unique_tasks_flagged", "unique_tasks", "accuracy"]].round(3)

Re-applied rules match the Exercise 05 validation predictions
Saved predictions for 526 validation assessments


,flagged (of 21 overdue)
logistic_regression,21
days_until_start_rule,21
days_rule_by_employee_type,21


,at_risk,TP,FN,recall,precision,flagged_assessments,flagged_%,unique_tasks_flagged,unique_tasks,accuracy
logistic_regression,84,78,6,0.929,0.451,173,34.257,129,397,0.800
days_until_start_rule,84,80,4,0.952,0.349,229,45.347,171,397,0.697
days_rule_by_employee_type,84,79,5,0.940,0.345,229,45.347,175,397,0.693


## F. Interpretation

**Data, split, and leakage checks.** All 16 checks passed: the data and saved split matched Exercise 05, every feature is a documented model input with no missing values, no identifier, label, outcome, simulation-only, or stratification field was used, no employee appears in more than one split or on both sides of a cross-validation fold, every row used for fitting came from the training split, and the re-applied baseline rules reproduced the Exercise 05 predictions. These checks provide evidence that the implemented workflow respects the training boundary; they do not prove that all possible leakage is absent.

**Validation comparison.** On this validation split, the model met the recall target while flagging 56 fewer assessments and 42 fewer unique tasks than the global rule, but missed two additional at-risk assessments. It reached 92.9% recall (78 of 84) and 45.1% precision, flagging 173 of 505 early-warning assessments (34.3%) across 129 tasks, compared with 95.2% recall, 34.9% precision, and 229 flagged assessments across 171 tasks for the global days rule. All 21 overdue validation assessments were flagged by the model and both days rules.

**Coefficients.** Days until start has the largest absolute coefficient among the standardized numeric inputs (−5.17 per standard deviation, about 7.5 business days): assessments with more days remaining had lower estimated risk. An open status, compared with in progress, and higher team workload were associated with higher estimated risk. Numeric and categorical coefficients describe different kinds of change and should not be ranked against each other by size. Some categorical coefficients differ in direction from the single-input rates in Exercise 04; for example, Parks has a negative coefficient compared with Administration. This may be because each coefficient is conditional on the other inputs, but this analysis does not establish exactly why.

**What this does not show.** These results describe a model applied to simulated outcomes, not real employees or HR performance, and the coefficients describe associations in the simulation, not causes of delays. One comparison on a small validation set clustered by task and employee is not a reliable estimate of improvement, and the two additional missed at-risk assessments represent a trade-off that Exercise 07 will examine when considering the consequences of different errors. The threshold was chosen on training data only; in Exercise 09, both the model and the baseline rules can be re-tuned on validation before one final test evaluation.